# 06.5 — MIL softmax pooling

The classifier produces a prediction *per window* (the `W`/`T` axis, [02.2](../02_numpy_and_pytorch_basics/02.2_array_axis_conventions.ipynb)), but the label is *per trial* — one dimension value for the whole trial, not one per window. **Multiple Instance Learning** (MIL) bridges the gap: it treats a trial as a *bag* of window-instances and pools them into a single per-trial prediction. This project's MIL uses a joint softmax across the time and class axes, then marginalizes. This notebook is about that pooling.

This notebook covers:

* The bag-of-instances framing: per-window predictions, per-trial label.
* Joint softmax over (time × class), then marginalize over time.
* Why this differs from just averaging the per-window softmaxes.
* The production `mil_multi_head_cross_entropy`.

**Prerequisites:** [04.6 multi-head classifier](../04_architecture/04.6_multi_head_classifier.ipynb), [06.1 overview](06.1_multi_task_losses_overview.ipynb).

## Section 1 — What MATLAB does

`cgg_softmaxLayer('SCT')` applies a joint softmax across the Spatial/Channel/Time axes, and `cgg_getPredictionFromClassifierProbabilities` aggregates:

```matlab
Y = softmax(logits, 'SCT');            % joint softmax over the pooled axes
Y_trial = sum(Y, [S, T]);              % marginalize → per-trial class distribution
loss = -log(Y_trial(target));          % CE on the marginal
```

`MultipleInstanceLearningType = 'MIL'` turns this on. The synthetic data has no spatial axis, so it's a joint softmax over Time × Class, then a sum over Time — leaving a per-trial marginal over the class axis. The port reproduces this precisely.

## Section 2 — The Python concepts you need

### 2.1 — The bag framing

A trial is a *bag* of `T` window-instances. The model scores each window for each class, giving `(T, K)` logits. But the trial has ONE label. MIL's job: pool the `(T, K)` scores into a single `(K,)` per-trial distribution to compare against that one label. The naive approach — average the per-window softmaxes — treats every window equally. MIL's joint softmax does something subtler.

### 2.2 — Joint softmax vs per-window softmax

In [ ]:
import torch
import torch.nn.functional as F

# Per-window logits for one trial: T=3 windows, K=2 classes.
logits = torch.tensor([[2.0, 0.0],      # window 0: leans class 0
                       [0.0, 0.0],      # window 1: uncertain
                       [0.0, 3.0]])     # window 2: strongly class 1

# Approach A — per-window softmax, then average over windows:
per_window = F.softmax(logits, dim=1)           # each ROW sums to 1
avg_softmax = per_window.mean(dim=0)            # (K,) — average distribution
print("per-window softmax then average:", [round(v, 3) for v in avg_softmax.tolist()])

# Approach B — MIL joint softmax over ALL (T×K) cells, then sum over T:
joint = F.softmax(logits.reshape(-1), dim=0).reshape(3, 2)   # ALL 6 cells sum to 1
marginal = joint.sum(dim=0)                     # (K,) — marginal over time
print("MIL joint softmax then marginalize:", [round(v, 3) for v in marginal.tolist()])

The two give different distributions. The key difference: per-window-then-average normalizes *within each window* (every window contributes equally, even the uncertain one). The joint softmax normalizes across *all* time×class cells at once, so a window with a large-magnitude logit (window 2's strong class-1 vote) dominates — confident windows count more. For MIL, that's the right behavior: a trial's evidence should come from its *informative* windows, not be diluted by uncertain ones.

### 2.3 — The marginal sums to 1

In [ ]:
# After the joint softmax, all T×K cells sum to 1; marginalizing over T
# leaves a proper probability distribution over K:
print("joint (all cells) sum:", joint.sum().item())
print("marginal over classes sum:", marginal.sum().item(), "(a valid per-trial distribution)")
print("→ CE compares this per-trial marginal against the trial's single label.")

The marginal is a valid distribution over the `K` classes for the whole trial — exactly what cross-entropy needs against the single per-trial target. So MIL turns `(T, K)` per-window logits into one `(K,)` per-trial prediction, and the CE is computed once per trial, not once per window.

### 2.4 — The production loss, live

In [ ]:
from neural_data_decoding.training.losses.classification import mil_multi_head_cross_entropy

# Two output dimensions, batch of 4 trials, T=3 windows each.
logits_dim0 = torch.randn(4, 3, 2)      # (B, T, K=2)
logits_dim1 = torch.randn(4, 3, 3)      # (B, T, K=3)
targets = torch.tensor([[0, 2], [1, 0], [0, 1], [1, 2]])   # (B, num_dims)

loss = mil_multi_head_cross_entropy([logits_dim0, logits_dim1], targets)
print("MIL multi-head CE loss:", round(loss.item(), 4))
print("→ joint softmax over (T, K) per dim, marginalize over T, CE on the marginal, summed over dims.")

One scalar, computed by pooling each dimension's `(B, T, K)` logits to `(B, K)` marginals and running cross-entropy against the per-trial targets. The multi-head structure ([04.6](../04_architecture/04.6_multi_head_classifier.ipynb)) is preserved — one MIL-pooled CE per dimension, summed.

### 2.5 — Why MIL for neural decoding

Not every window in a trial carries the decoding signal — the informative moment might be a few windows near stimulus onset, with the rest noise. MIL lets the model *learn which windows matter* rather than forcing every window to predict the label. That's a better fit for neural data than demanding a correct prediction at every timestep, and it's why the Optimal config sets `MultipleInstanceLearningType='MIL'`. The alternative (`'None'`) computes per-window CE directly — [06.11](06.11_single_total_loss_three_subnetworks.ipynb)'s interpolated CE blends the two.

## Section 3 — The `neural_data_decoding` implementation

`mil_multi_head_cross_entropy` — the joint-softmax-then-marginalize, per dimension:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/losses/classification.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def _mil_marginal_probs"))
for k in range(i, min(i + 15, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `_mil_marginal_probs` reshapes `(B, T, K)` → `(B, T*K)`, softmaxes over the flattened axis (the joint softmax, §2.2), reshapes back, and sums over `T` — the marginal.
* The docstring cites `cgg_softmaxLayer('SCT')` and the exact MATLAB aggregation line — parity traced to the source.
* A small `eps` clamps the marginal before `log` to avoid `log(0)` ([02.8](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb) territory — a near-zero probability would produce `-inf`).
* The per-dimension loop mirrors [04.6](../04_architecture/04.6_multi_head_classifier.ipynb)'s list-of-logits — MIL pools each dimension independently, then sums the CEs.

## Section 4 — Hands-on exercises

### Exercise 4.1 — confident windows dominate

Build a trial where one window is very confident (large logits) and the rest are uncertain (near-zero logits). Show the MIL marginal follows the confident window, while per-window-average is diluted.

In [ ]:
# Reveal:
logits = torch.tensor([[0.1, 0.0], [0.0, 0.1], [5.0, 0.0]])   # window 2 SHOUTS class 0
avg = F.softmax(logits, dim=1).mean(dim=0)
joint = F.softmax(logits.reshape(-1), dim=0).reshape(3, 2)
mil = joint.sum(dim=0)
print("per-window average:", [round(v, 3) for v in avg.tolist()], "(diluted toward 0.5)")
print("MIL marginal:      ", [round(v, 3) for v in mil.tolist()], "(follows the confident window)")

### Exercise 4.2 — the marginal is a distribution

For random `(B, T, K)` logits, confirm each trial's MIL marginal sums to 1 (a valid distribution).

In [ ]:
# Reveal:
logits = torch.randn(5, 4, 3)          # B=5, T=4, K=3
B, T, K = logits.shape
joint = F.softmax(logits.reshape(B, T*K), dim=1).reshape(B, T, K)
marginal = joint.sum(dim=1)            # (B, K)
print("per-trial marginal sums:", [round(v, 4) for v in marginal.sum(dim=1).tolist()])

## Section 5 — Common errors

### MIL and per-window CE give very different results

Expected — they pool differently (§2.2). MIL is not "average the softmaxes." Which to use is the `MultipleInstanceLearningType` config choice; [06.11](06.11_single_total_loss_three_subnetworks.ipynb)'s interpolated CE blends them.

### `-inf` or NaN loss from MIL

A marginal probability hit exactly 0 before `log`. The `eps` clamp (§3) prevents it; if you rolled your own MIL, add the clamp.

### The time axis is missing

MIL's whole point is pooling over `T` — logits must be `(B, T, K)`, not `(B, K)`. A classifier that collapsed the window axis ([04.5 §2.2](../04_architecture/04.5_the_bottleneck.ipynb) warns against this) leaves nothing to pool.

### Softmax over the wrong axis

The joint softmax is over the *flattened* `(T×K)`, not over `K` alone or `T` alone. Softmaxing over `K` per-window then summing is the per-window-average approach, a different loss.

### MIL slower than expected

The reshape + joint softmax over `T×K` is slightly more work than per-window CE, but negligible. If MIL dominates runtime, something else (data loading, [03.2](../03_data_pipeline/03.2_dataloader_and_collation.ipynb)) is the real bottleneck.

## Section 6 — Further reading

- [Attention-based Deep MIL (Ilse et al.)](https://arxiv.org/abs/1802.04712) — the modern MIL framing.
- [`src/neural_data_decoding/training/losses/classification.py`](../../src/neural_data_decoding/training/losses/classification.py) — `mil_multi_head_cross_entropy` + `_mil_marginal_probs`.
- [`tests/unit/test_mil_softmax.py`](../../tests/unit/test_mil_softmax.py) / `test_mil_cross_entropy.py` — the parity tests.

Next up: **[06.6 confidence routing](06.6_confidence_routing.ipynb)** — the Trial and Task confidence heads and what each outputs.